# Customizing LLMs with Amazon SageMaker AI — A Beginner's Hands-On Lab

**What you will do in this notebook:**
1. Teach a 4-billion-parameter model to review NDAs like a legal checklist (**supervised fine-tuning with LoRA**)
2. Verify the improvement with a **managed evaluation** that scores the base and fine-tuned model side by side
3. **Deploy** your fine-tuned model and talk to it over an OpenAI-compatible API
4. Teach a 1B chat model to sound human instead of robotic (**DPO — Direct Preference Optimization**)
5. Measure that with an **LLM-as-a-judge** evaluation

**How you pay:** training is *serverless* — you never pick an instance, and you are billed per token
processed. The whole lab costs **roughly $5–10** if you delete resources when told to.

**Where it runs:** in **SageMaker Studio** (JupyterLab) or on your own machine with the AWS CLI configured.
Use region `us-east-1`, `us-west-2`, `eu-west-1`, or `ap-northeast-1` (serverless customization is
available only in these regions).

## 🗺️ The big picture — what this notebook does, end to end

You will take an open model (**Qwen3-4B**) that knows nothing about legal contracts, teach it to
review NDAs like a lawyer's checklist (**SFT with LoRA**), prove the improvement with a
side-by-side comparison, serve it as a real API, and then teach a second model (**Llama 3.2 1B**)
to sound human instead of corporate (**DPO**).

```mermaid
flowchart LR
    A[📥 Part 1<br/>607 real NDAs<br/>from Stanford] --> B[🧠 Part 2<br/>serverless LoRA<br/>fine-tuning, ~30 min]
    B --> C[⚖️ Part 3<br/>deploy base + tuned<br/>side-by-side A/B]
    C --> D[💬 Part 4<br/>talk to it via API<br/>+ teardown]
    D --> E[🎭 Part 5<br/>DPO tone training<br/>+ tone test]
    E --> F[🧹 Teardown<br/>back to ~0 cost]
```

| Part | What you pay | Wall clock |
|---|---|---|
| 1 — Data | pennies (S3 storage) | ~5 min |
| 2 — SFT training | ~$2 (pay-per-token, no GPU rental) | ~30 min |
| 3 — A/B evaluation | endpoint time only (~$1/hr × 2, for minutes) | ~25 min incl. deploys |
| 4 — Inference + teardown | ~$0.30 | ~5 min |
| 5 — DPO + tone test | ~$1–2 | ~35 min |

> 🧭 **How to read the cells:** every code cell starts with a comment telling you what it does
> and what it costs. `▶ RUN` = safe to execute. `🛑 BEFORE YOU RUN` = starts billing — read first.

## Part 0 — Setup & safety checks (costs nothing)

We verify everything **before** spending a cent. Beginners rarely fail on ML concepts —
they fail on permissions, quotas, and missing resources. These five checks prevent that.

In [3]:
# ▶ RUN — installs the libraries used throughout the lab (~2 min, no cost)
%pip install  "sagemaker>=3.16" boto3 datasets==5.0.0 pandas scikit-learn pyarrow tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.


### 0.1 — Create a session and find your execution role

In SageMaker Studio, `get_execution_role()` returns your domain's role automatically.
On your own machine, set the `ROLE_ARN` environment variable instead (see the repo's
`.env.example`).

In [1]:
# ▶ RUN — no cost
import os
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
REGION = sess.boto_region_name
BUCKET = sess.default_bucket()

try:
    ROLE_ARN = get_execution_role()          # works inside SageMaker Studio
    print("Running in Studio — using domain execution role")
except ValueError:
    ROLE_ARN = os.environ.get("ROLE_ARN", "") # works on your local machine
    if not ROLE_ARN:
        raise SystemExit("Set the ROLE_ARN env var (copy .env.example to .env). See the repo README.")

print(f"region : {REGION}")
print(f"bucket : {BUCKET}")
print(f"role   : {ROLE_ARN}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Running in Studio — using domain execution role
region : us-east-1
bucket : sagemaker-us-east-1-685057748560
role   : arn:aws:iam::685057748560:role/service-role/AmazonSageMaker-ExecutionRole-20261005T173732


> ⚠️ **Common error — `Could not assume role .../AmazonSageMaker-ExecutionRole-...`**
> Console-created roles live under a `/service-role/` path. The ARN must include it:
> `arn:aws:iam::<account>:role/service-role/AmazonSageMaker-ExecutionRole-...`
> Run `aws iam get-role --role-name <role-name>` to see the exact ARN.

### 0.2 — Good to know before you spend anything

- **One training job at a time.** Serverless customization runs one job per region by
  default (quota = 1). This notebook already runs things sequentially — just don't
  launch Part 2 and Part 5 at the same time.
- **No evaluation quota or MLflow server needed.** Part 3 evaluates with a local
  side-by-side comparison, and Part 5.1 with a tone test. (The managed evaluation
  pipelines — which do need a quota increase in new accounts — are the optional
  at-scale path; see `scripts/03_evaluate_sft.py` and `scripts/06_evaluate_dpo.py`.)
- **Region** must be a supported one — you confirmed yours in the cell above
  (`us-east-1` ✅). Running locally instead of Studio? Then copy `.env.example` to
  `.env` and set `ROLE_ARN` (see the repo README).

> 🛑 **Budget alarm (2 minutes, worth it):** AWS Budgets → create a monthly cost budget
> (e.g. $20) with an 80% email alert. Serverless billing lags by hours, so the console
> can show $0 while you're actually spending.

## Part 1 — The task and the data (SFT)

**The mission:** build a model that reads a Non-Disclosure Agreement and answers a fixed
17-point legal checklist — *"Is there a non-disclosure obligation? Which paragraph proves it?"* —
returning strict JSON with a verdict and evidence spans.

**The dataset:** **ContractNLI** — 607 real NDAs annotated by legal experts (CC-BY-4.0).
The base model gets only ~65% of checklist items right, so there's real headroom to improve.

### 1.1 — Load the dataset

We reuse a helper module from the official AWS workshop repo (`contractnli.py`), which
downloads the dataset and builds prompts. If you don't have the repo yet, the cell clones it.

**Where the data comes from (no dataset ships with this repo):** the notebook downloads
**ContractNLI** — 607 real non-disclosure agreements annotated by legal experts (Stanford, CC-BY-4.0) —
the first time you run this cell, then formats it into the shape SageMaker expects.

```mermaid
flowchart LR
    A[🌐 Stanford<br/>contract-nli.zip] -->|download once| B[📁 ./data/contract-nli<br/>raw documents +<br/>expert annotations]
    B -->|next cells:<br/>format into<br/>prompt/completion pairs| C[📄 dataset.jsonl]
    C -->|upload| D[🪣 S3 bucket]
    D -->|register, one line| E[📚 SageMaker AI Registry<br/>versioned datasets]
```

In [2]:
# ▶ RUN — downloads the dataset on first run (~1 min, no cost)
import pathlib, subprocess, sys

WORKSHOP_DIR = pathlib.Path(os.environ.get(
    "WORKSHOP_DIR", "~/generative-ai-on-amazon-sagemaker")).expanduser()
LAB1 = WORKSHOP_DIR / "workshops/serverless-model-customization-with-sagemaker-ai/lab-1-supervised-fine-tuning"

if not (LAB1 / "contractnli.py").exists():
    print("Workshop repo not found — cloning it now...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/aws-samples/generative-ai-on-amazon-sagemaker.git",
                    str(WORKSHOP_DIR)], check=True)

sys.path.insert(0, str(LAB1))
import contractnli as C   # the workshop's helper module

C.ensure_dataset("./data")
train_docs, labels = C.load("train")
dev_docs, _ = C.load("dev")
test_docs, _ = C.load("test")
print(f"train {len(train_docs)} contracts | dev {len(dev_docs)} | test {len(test_docs)}")
print(f"checklist items: {len(labels)}")

Workshop repo not found — cloning it now...


Cloning into '/home/sagemaker-user/generative-ai-on-amazon-sagemaker'...


downloading ContractNLI from https://stanfordnlp.github.io/contract-nli/resources/contract-nli.zip ...
unpacked to data/contract-nli
train 423 contracts | dev 61 | test 123
checklist items: 17


> ⚠️ **Common error — `ModuleNotFoundError: No module named 'contractnli'`**
> The `WORKSHOP_DIR` path doesn't point at the aws-samples repo. Check the clone succeeded
> and the path exists.

✅ **Checkpoint:** you should see `train 423 contracts | dev 61 | test 123` and `checklist items: 17`.

### 1.2 — Look at one example

Understanding *one* example deeply beats skimming a thousand. This is a real contract
from the dataset, its clause split, and the expert's gold answer.

In [3]:
# ▶ RUN — no cost
import json

doc = sorted(test_docs, key=lambda d: len(d["text"]))[1]   # a short-ish one
gold = C.gold_for(doc)

print(f"{doc['file_name']} — {len(doc['text'].split())} words\n")
print("A few clauses:")
for number, text in C.doc_spans(doc)[:3]:
    print(f"  [{number}] {text[:110]}...")

print("\nGold answer (what we want the model to produce):")
for k in sorted(gold, key=lambda x: int(x.split('-')[1]))[:5]:
    v = gold[k]
    ev = f"  evidence clauses {v['spans']}" if v["spans"] else ""
    print(f"  {k}  {v['choice']:14s}{ev}   [{labels[k]['short_description']}]")

1023734_0000912057-96-023266_document_16.txt — 232 words

A few clauses:
  [0] NAVIDEC, INCORPORATED...
  [1] TRADE SECRET/NON-DISCLOSURE AGREEMENT...
  [2] In consideration of the mutual promises made herein, as well as the agreement between Navidec, Incorporated an...

Gold answer (what we want the model to produce):
  nda-1  NotMentioned     [Explicit identification]
  nda-2  Contradiction   evidence clauses [3, 4]   [None-inclusion of non-technical information]
  nda-3  NotMentioned     [Inclusion of verbally conveyed information]
  nda-4  Entailment      evidence clauses [4]   [Limited use]
  nda-5  Contradiction   evidence clauses [3]   [Sharing with employees]


### 1.3 — Build the training records

SageMaker AI's training service expects **prompt/completion JSONL** — one JSON object per line:

```json
{"prompt": "<full contract + checklist + output rules>", "completion": "{"checklist-1": {"label": "Entailment", ...}}"}
```

> 🎓 **Glossary — JSONL:** "JSON Lines". Each line is one independent JSON object. Streaming-friendly,
> the standard format for LLM training data.
>
> ⚠️ **Why formatting is where fine-tuning projects die:** if your records are malformed
> (wrong field names, nested JSON as a string, wrong chat schema), training *runs* — it just
> learns nothing. Always eyeball a record before paying for training.

The evaluation split uses a different schema (`query`/`response`) because the managed scorer
reads that shape — the helper handles it for us.

**The format SageMaker's trainer expects — `prompt` / `completion`:** every training example is a
conversation. The *prompt* is what a user would send (here: the full contract + the 17-point checklist
+ strict instructions). The *completion* is the ideal answer (here: the legal expert's verdicts as JSON).
Fine-tuning teaches the model to produce completions like these for prompts like these.

```mermaid
flowchart TB
    subgraph one["One training record"]
        P["📝 prompt<br/><br/>SYSTEM: You are a contract review assistant...<br/>USER: [full NDA text] + checklist of 17<br/>hypotheses + 'respond with JSON only'"]
        C["✅ completion<br/><br/>{#quot;nda-1#quot;: {#quot;label#quot;: #quot;Entailment#quot;, #quot;evidence#quot;: [4]},<br/>#quot;nda-2#quot;: {#quot;label#quot;: #quot;Contradiction#quot;}, ...}"]
        P --> C
    end
```

> 💡 One detail that matters: these contracts average **~15,000 characters (~3,700 tokens)** per record.
> The trainer's default 4k token window would silently truncate a third of the dataset — that's why
> Part 2 sets `dataset_max_len = 16384`.

In [4]:
# ▶ RUN — no cost
label_keys = list(labels.keys())

def gold_json(doc):
    g = C.gold_for(doc)
    return json.dumps({k: {"label": g[k]["choice"], "evidence": list(g[k]["spans"])}
                       for k in label_keys if k in g})

records = {
    "train": [{"prompt": C.build_prompt(d, labels, no_think=False), "completion": gold_json(d)} for d in train_docs],
    "val":   [{"prompt": C.build_prompt(d, labels, no_think=False), "completion": gold_json(d)} for d in dev_docs],
    "test":  [{"query": C.build_prompt(d, labels), "response": gold_json(d)} for d in test_docs],
}

for name, rows in records.items():
    field = "query" if name == "test" else "prompt"
    avg = sum(len(r[field]) for r in rows) // len(rows)
    print(f"{name:5s}: {len(rows):4d} records, avg prompt {avg:,} chars (~{avg//4:,} tokens)")

train:  423 records, avg prompt 14,650 chars (~3,662 tokens)
val  :   61 records, avg prompt 15,730 chars (~3,932 tokens)
test :  123 records, avg prompt 14,854 chars (~3,713 tokens)


✅ **Checkpoint:** `train: 423 records`, `val: 61`, `test: 123`, average prompts of several
thousand characters. **Notice the token estimate** — it matters next:

> 🎓 **Why we override `dataset_max_len` later:** contracts are long. With the recipe default
> of 4096 tokens, the trainer would *silently drop* any record that doesn't fit — about a
> third of our dataset. We set 16384 so nothing is discarded.

### 1.4 — Upload to S3 and register in the SageMaker AI Registry

The **AI Registry** versions your datasets so every training run is reproducible:
`DataSet.create(name=..., source=s3://..., customization_technique=SFT)`.

In [ ]:
# ▶ RUN — uploads ~2 MB to S3, registers 3 datasets (~2 min, pennies at most)
import shutil
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

DATASET_PREFIX = "contractnli-nda-review"
s3_client = boto3.client("s3")
default_prefix = sess.default_bucket_prefix

# --- Step 1/3: write the formatted records to local JSONL files -----------------
local = pathlib.Path("./sft_data")
if local.exists():
    shutil.rmtree(local)
for name, rows in records.items():
    d = local / name
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "dataset.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
print(f"Step 1/3 ✅ wrote local JSONL: {sum(len(v) for v in records.values())} records "
      f"({', '.join(records)})")

# --- Step 2/3: upload the files to your S3 bucket -------------------------------
input_path = (f"{default_prefix}/datasets/{DATASET_PREFIX}" if default_prefix
              else f"datasets/{DATASET_PREFIX}")
s3_paths = {}
for name in records:
    key = f"{input_path}/{name}/dataset.jsonl"
    s3_client.upload_file(str(local / name / "dataset.jsonl"), BUCKET, key)
    s3_paths[name] = f"s3://{BUCKET}/{key}"
    print(f"Step 2/3 ⬆ uploaded {name}: {s3_paths[name]}")

# --- Step 3/3: register each split in the SageMaker AI Registry -----------------
# The Registry versions your data, so any training run can point at an exact dataset
# version and be reproduced later.
training_dataset = DataSet.create(name=f"{DATASET_PREFIX}-train", source=s3_paths["train"],
                                  customization_technique=CustomizationTechnique.SFT,
                                  wait=True, role=ROLE_ARN)
print("Step 3/3 📚 registered train (SFT tag)")
val_dataset = DataSet.create(name=f"{DATASET_PREFIX}-val", source=s3_paths["val"],
                             customization_technique=CustomizationTechnique.SFT,
                             wait=True, role=ROLE_ARN)
test_dataset = DataSet.create(name=f"{DATASET_PREFIX}-test", source=s3_paths["test"],
                              wait=True, role=ROLE_ARN)
print("\n✅ 3 datasets registered in the AI Registry — training can now reference them by name")

## Part 2 — Serverless LoRA fine-tuning

> 🎓 **Glossary — LoRA / PEFT:** a 4B model has ~4 billion weights. Full fine-tuning updates
> all of them (huge memory, huge cost). **LoRA** freezes the base model and trains two tiny
> "adapter" matrices per layer (rank 32 here — millions, not billions, of parameters). The
> adapters capture the task; at serving time they're merged back into the base weights.
> **PEFT** = Parameter-Efficient Fine-Tuning, the family of techniques LoRA belongs to.

> 🎓 **What "serverless training" means:** you don't choose an instance type, don't wait for
> capacity, don't configure a cluster. SageMaker AI provisions compute sized to your model
> and dataset, and bills **per token processed**. When the job ends, the compute disappears.

**Hyperparameters we use (and why):**

| Setting | Value | Why |
|---|---|---|
| `lora_rank` / `lora_alpha` | 32 / 64 | adapter size vs. adapter learning scale — a solid default for 4B models |
| `max_epochs` | 8 | 423 records ÷ batch 64 ≈ 7 steps/epoch → 52 optimizer steps total |
| `global_batch_size` | 64 | memory/throughput trade-off; serverless default is fine |
| `dataset_max_len` | 16384 | so long contracts aren't silently dropped (see 1.3) |

In [6]:
# ▶ RUN — creates the model package group, no cost
import hashlib
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

BASE_MODEL_ID = "huggingface-reasoning-qwen3-4b"   # Qwen3-4B

MAX_MPG = 63
suffix = "-contractnli-sft-mpg"
candidate = f"{BASE_MODEL_ID}{suffix}"
if len(candidate) > MAX_MPG:
    digest = hashlib.sha1(BASE_MODEL_ID.encode()).hexdigest()[:6]
    keep = MAX_MPG - len(suffix) - len(digest) - 1
    MPG = f"{BASE_MODEL_ID[:keep].rstrip('-')}-{digest}{suffix}"
else:
    MPG = candidate

try:
    ModelPackageGroup.get(model_package_group_name=MPG)
    print(f"model package group exists: {MPG}")
except ClientError:
    ModelPackageGroup.create(model_package_group_name=MPG,
                             model_package_group_description="ContractNLI NDA review, serverless SFT")
    print(f"created model package group: {MPG}")

[10/05/26 12:41:26] WARNING  No region provided. Using default region.                                 ]8;id=3235057;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=3235058;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#350\350]8;;\

model package group exists: huggingface-reasoning-qwen3-4b-contractnli-sft-mpg


✅ Checkpoint: you should see `created model package group: huggingface-reasoning-qwen3-4b-contractnli-sft-mpg`
(or `... exists` on a re-run).

**What LoRA actually does (and why serverless makes it cheap):** full fine-tuning updates all
~4 billion weights of Qwen3-4B. **LoRA freezes the base model** and trains only tiny "adapter" matrices
inserted into each layer — here, a few million parameters instead of billions. That's why one API call
can train it serverlessly and bill per token.

```mermaid
flowchart LR
    subgraph full["Full fine-tuning ❌ expensive"]
        A1["4B weights<br/>all updated"]
    end
    subgraph lora["LoRA ✅ what we do"]
        B1["4B weights<br/>FROZEN"] --- B2["small adapter matrices<br/>trained: r=32, alpha=64<br/>≈ millions of params"]
    end
```

**The hyperparameters you'll set below, in plain language:**

| Setting | Value | Meaning |
|---|---|---|
| `global_batch_size` | 64 | contracts processed between weight updates |
| `max_epochs` | 8 | passes over the 423 training contracts (more passes = more memorization, watch for overfitting) |
| `learning_rate` | 0.0001 | how big each adjustment step is — LoRA adapters learn gently |
| `lr_warmup_steps_ratio` | 0.1 | first 10% of steps ramp the rate up (stability) |
| `lora_rank` / `lora_alpha` | 32 / 64 | adapter size / adapter scaling — the two knobs that control LoRA capacity |
| `dataset_max_len` | 16384 | token window per record — must fit our ~3.7k-token contracts with headroom |

**And "serverless" means:** no `instance_type="ml.…"` anywhere. SageMaker provisions GPU compute when the
job starts and releases it at the end — you pay per token processed, not per GPU-hour.

In [ ]:
# 🛑 BEFORE YOU RUN — this starts paid training (~$2, ~29 min). Make sure Part 0 checks were green.
# ▶ RUN — launches the serverless SFT job
output_path = (f"s3://{BUCKET}/{default_prefix}/{BASE_MODEL_ID}-contractnli"
               if default_prefix else f"s3://{BUCKET}/{BASE_MODEL_ID}-contractnli")

trainer = SFTTrainer(
    model=BASE_MODEL_ID,
    training_type=TrainingType.LORA,
    model_package_group=MPG,
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=ROLE_ARN,
    accept_eula=True,          # required for JumpStart models
    base_job_name="contractnli-sft",
)

trainer.hyperparameters.global_batch_size = 64      # contracts per weight update
trainer.hyperparameters.max_epochs = 8              # 8 full passes over the 423 contracts
trainer.hyperparameters.learning_rate = 0.0001      # LoRA adapters learn gently
trainer.hyperparameters.lr_warmup_steps_ratio = 0.1 # first 10% of steps: ramp rate up
trainer.hyperparameters.lora_rank = 32              # adapter size (capacity knob)
trainer.hyperparameters.lora_alpha = 64             # adapter scaling (2x rank here)
trainer.hyperparameters.dataset_max_len = 16384     # token window: fits ~3.7k-token contracts

print("hyperparameters we set (see the table above for what each means):")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

training_job = trainer.train(wait=False)
JOB_NAME = training_job.training_job_name
print(f"\nLAUNCHED: {JOB_NAME}")

In [8]:
# ▶ RUN — watch the job (safe to re-run; also safe to close the laptop and come back)
import time
from sagemaker.core.resources import TrainingJob

while True:
    j = TrainingJob.get(training_job_name=JOB_NAME)
    print(f"{time.strftime('%H:%M:%S')}  {j.training_job_status} / {j.secondary_status}")
    if j.training_job_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(120)

12:44:07  InProgress / Starting
12:46:07  InProgress / Downloading
12:48:07  InProgress / Downloading
12:50:07  InProgress / Training
12:52:07  InProgress / Training
12:54:07  InProgress / Training
12:56:07  InProgress / Training
12:58:08  InProgress / Training
13:00:08  InProgress / Training
13:02:08  InProgress / Training
13:04:08  InProgress / Training
13:06:08  InProgress / Training
13:08:08  InProgress / Training
13:10:08  InProgress / Training
13:12:08  InProgress / Uploading
13:14:08  Completed / Completed


**What the statuses mean:** `Downloading` (fetching the base model) → `Training` (the
actual LoRA pass — loss curves are being logged to MLflow automatically) → `Uploading`
(artifacts to S3) → `Completed`.

📊 **Reference results** (measured in a real run of this exact configuration):
**~29 minutes wall-clock**, checklist accuracy **64.5% → 87.4%**, evidence F1 **48.8 → 75.4**
on the 123 held-out contracts. Your numbers will differ slightly — that's fine and a good
discussion point.

## Part 3 — Did it actually work? Evaluation, two ways

**Never trust training loss alone.** Two ways to check the fine-tuned model, and we'll do both:

**1. The reference numbers.** Measured properly, with managed evaluation pipelines — a custom scorer checks every verdict AND whether the cited evidence clauses match the gold annotation, scoring base and fine-tuned models on the same 123 held-out contracts. The workshop's published result: checklist accuracy jumps from **64.5% to 87.4%**, evidence F1 from **48.8 to 75.4**. Those pipelines need an evaluation-quota increase and an MLflow tracking server (the ⚠️ items in Part 0) — so today we prove the improvement live, the do-it-yourself way.

**2. The live proof (no quota needed).** Deploy **both** the base Qwen3-4B and your fine-tuned model, ask them the **same question about the same real contract**, and score the answers with the exact scorer logic the workshop uses — running locally in this notebook.

> 💡 **At scale later:** once your account has evaluation quota, the managed path (register the scorer as a reward function, one pipeline job scores both models) is the repeatable option — see `scripts/03_evaluate_sft.py` in this repo.

**Why the deployment takes four API calls (this confuses everyone at first):** serving a model on
SageMaker is a chain of four objects, each with one job:

```mermaid
flowchart LR
    M["1️⃣ Model<br/>the recipe:<br/>weights in S3 +<br/>container image"] --> C["2️⃣ EndpointConfig<br/>the hardware plan:<br/>ml.g5.xlarge ×1"]
    C --> E["3️⃣ Endpoint<br/>the actual GPU<br/>machine (up in ~2 min)"]
    E --> IC["4️⃣ InferenceComponent<br/>downloads weights into the<br/>container, loads vLLM (~7–9 min)"]
```

| Call | Creates | Analogy |
|---|---|---|
| `Model.create` | a named bundle: *where the weights live* (S3) + *what serves them* (DJL container running vLLM) | the recipe |
| `EndpointConfig.create` | the hardware plan: instance type (`ml.g5.xlarge`), count, timeouts | the kitchen order |
| `Endpoint.create` | the actual machine — this is what bills ~$1/hour while it exists | the kitchen |
| `InferenceComponent.create` | loads your model into the machine and makes it reachable | the cook + the dish |

The cell below prints `Step 1/4 … Step 4/4` so you can watch each stage happen. Billing starts at
Step 2/3 (the Endpoint) — that's the `🛑` warning.

In [ ]:
# 🛑 BEFORE YOU RUN — endpoint billing starts NOW (~$1/hr per endpoint until teardown).
# ▶ RUN — deploys the FINE-TUNED model (~10–15 min): model, config, endpoint, inference component


from sagemaker.core import s3 as sm_s3
from sagemaker.core.resources import (Endpoint, EndpointConfig, InferenceComponent, Model)
from sagemaker.core.shapes import (ContainerDefinition, InferenceComponentComputeResourceRequirements,
                                   InferenceComponentRuntimeConfig, InferenceComponentSpecification,
                                   ModelDataSource, ProductionVariant, S3ModelDataSource)

sm = boto3.client("sagemaker", region_name=REGION)
resp = sm.list_model_packages(ModelPackageGroupName=MPG, SortBy="CreationTime",
                              SortOrder="Descending", MaxResults=1)
model_package_arn = resp["ModelPackageSummaryList"][0]["ModelPackageArn"]
print(f"fine-tuned model: {model_package_arn}")

model_package = __import__("sagemaker.core.resources", fromlist=["ModelPackage"]).ModelPackage.get(model_package_arn)
merged_uri = sm_s3.s3_path_join(
    model_package.inference_specification.containers[0]
    .model_data_source.s3_data_source.s3_uri, "checkpoints", "hf_merged") + "/"
print(f"merged checkpoint: {merged_uri}")

IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:0.36.0-lmi18.0.0-cu128"
MODEL_NAME, CFG_NAME, EP_NAME, IC_NAME = (f"{BASE_MODEL_ID[:40]}-contractnli-sft-{s}"
                                          for s in ["m", "cfg", "ep", "ic"])

env = {"HF_MODEL_ID": "/opt/ml/model", "OPTION_TRUST_REMOTE_CODE": "true",
       "OPTION_MODEL_LOADING_TIMEOUT": "3600", "OPTION_TENSOR_PARALLEL_DEGREE": "max",
       "SERVING_FAIL_FAST": "true", "OPTION_ROLLING_BATCH": "disable",
       "OPTION_ASYNC_MODE": "true", "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
       "OPTION_DTYPE": "bf16", "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 32)}

print("Step 1/4 🧩 Model — packaging weights (S3) + serving container (DJL/vLLM)...")
Model.create(model_name=MODEL_NAME,
             primary_container=ContainerDefinition(
                 image=IMAGE,
                 model_data_source=ModelDataSource(
                     s3_data_source=S3ModelDataSource(s3_uri=merged_uri, s3_data_type="S3Prefix",
                                                      compression_type="None")),
                 environment=env),
             execution_role_arn=ROLE_ARN)
print(f"      ✅ model created")
print("Step 2/4 🖥  EndpointConfig — the hardware plan: ml.g5.xlarge × 1...")
EndpointConfig.create(endpoint_config_name=CFG_NAME, execution_role_arn=ROLE_ARN,
                      production_variants=[ProductionVariant(
                          variant_name="AllTraffic", instance_type="ml.g5.xlarge",
                          initial_instance_count=1,
                          model_data_download_timeout_in_seconds=700,
                          container_startup_health_check_timeout_in_seconds=700)])
print("      ✅ endpoint config created (billing starts at the next step)\n")
print("Step 3/4 ⚡ Endpoint — starting the GPU machine (~2 min)...")
Endpoint.create(endpoint_name=EP_NAME, endpoint_config_name=CFG_NAME)
Endpoint.get(EP_NAME).wait_for_status("InService")
print("      ✅ endpoint InService — the machine is up\n")
print("Step 4/4 📦 InferenceComponent — downloading ~8 GB of weights into the container")
print("      and loading them into vLLM — the long step, ~7–9 min. This is a good moment")
print("      to open the endpoint in the SageMaker console and watch it happen.")
InferenceComponent.create(
    inference_component_name=IC_NAME, endpoint_name=EP_NAME, variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=MODEL_NAME,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=1024, number_of_accelerator_devices_required=1)),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1), region=REGION)
InferenceComponent.get(IC_NAME).wait_for_status("InService")
print("      ✅ inference component InService\n")
print(f"fine-tuned ready — sm:{EP_NAME}/{IC_NAME}@{REGION}")

In [ ]:
# 🛑 BEFORE YOU RUN — second endpoint = second ~$1/hr. (Endpoint quota for
#    ml.g5.xlarge defaults to 2, so both fit.)
# ▶ RUN — deploys the BASE Qwen3-4B from the JumpStart artifact cache (~10–15 min)
BASE_ARTIFACT = ("s3://jumpstart-cache-prod-us-east-1/huggingface-reasoning/"
                 "huggingface-reasoning-qwen3-4b/artifacts/inference/v1.0.0/")

BASE_MODEL_NAME, BASE_CFG, BASE_EP, BASE_IC = (f"{BASE_MODEL_ID[:40]}-contractnli-base-{s}"
                                               for s in ["m", "cfg", "ep", "ic"])

print("Step 1/4 🧩 Model — packaging weights (S3) + serving container (DJL/vLLM)...")
Model.create(model_name=BASE_MODEL_NAME,
             primary_container=ContainerDefinition(
                 image=IMAGE,
                 model_data_source=ModelDataSource(
                     s3_data_source=S3ModelDataSource(s3_uri=BASE_ARTIFACT, s3_data_type="S3Prefix",
                                                      compression_type="None")),
                 environment=env),
             execution_role_arn=ROLE_ARN)
print(f"      ✅ model created")
print("Step 2/4 🖥  EndpointConfig — the hardware plan: ml.g5.xlarge × 1...")
EndpointConfig.create(endpoint_config_name=BASE_CFG, execution_role_arn=ROLE_ARN,
                      production_variants=[ProductionVariant(
                          variant_name="AllTraffic", instance_type="ml.g5.xlarge",
                          initial_instance_count=1,
                          model_data_download_timeout_in_seconds=700,
                          container_startup_health_check_timeout_in_seconds=700)])
print("      ✅ endpoint config created (billing starts at the next step)\n")
print("Step 3/4 ⚡ Endpoint — starting the GPU machine (~2 min)...")
Endpoint.create(endpoint_name=BASE_EP, endpoint_config_name=BASE_CFG)
Endpoint.get(BASE_EP).wait_for_status("InService")
print("      ✅ endpoint InService — the machine is up\n")
print("Step 4/4 📦 InferenceComponent — downloading ~8 GB of weights into the container")
print("      and loading them into vLLM — the long step, ~7–9 min. This is a good moment")
print("      to open the endpoint in the SageMaker console and watch it happen.")
InferenceComponent.create(
    inference_component_name=BASE_IC, endpoint_name=BASE_EP, variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=BASE_MODEL_NAME,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=1024, number_of_accelerator_devices_required=1)),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1), region=REGION)
InferenceComponent.get(BASE_IC).wait_for_status("InService")
print("      ✅ inference component InService\n")
print(f"base ready — sm:{BASE_EP}/{BASE_IC}@{REGION}")

In [12]:
# ▶ RUN — the money shot: SAME contract, SAME prompt, two models (pennies)
import re
import time
from botocore.config import Config

C.ensure_dataset("./data")
doc = test_docs[0]
smr = boto3.client("sagemaker-runtime", region_name=REGION,
                   config=Config(read_timeout=300, retries={"total_max_attempts": 3}))
body = {"messages": [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]}
                     for m in C.build_messages(doc, labels)],
        "max_tokens": 4000, "temperature": 0.0, "stop": ["<|im_end|>"], "stream": False}

answers = {}
for which, ep, ic in [("BASE", BASE_EP, BASE_IC), ("FINE-TUNED", EP_NAME, IC_NAME)]:
    b = dict(body, model_name=ic)
    r = smr.invoke_endpoint(EndpointName=ep, InferenceComponentName=ic,
                            ContentType="application/json", Body=json.dumps(b))
    answers[which] = json.loads(r["Body"].read())["choices"][0]["message"]["content"]
    print(f"\n================ {which} ================")
    print(answers[which][:1200])

for which, text in answers.items():
    clean = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    pred, ok = __import__("contractnli_scorer").parse_answer(clean)
    n = len(pred) if isinstance(pred, dict) else 0
    print(f"{which}: valid JSON = {ok}, checklist items answered = {n} of {len(labels)}")


================ BASE ================
<think>

</think>

{
  "nda-1": {
    "label": "Entailment",
    "evidence": [24, 34]
  },
  "nda-2": {
    "label": "Contradiction",
    "evidence": [35]
  },
  "nda-3": {
    "label": "Entailment",
    "evidence": [34]
  },
  "nda-4": {
    "label": "Entailment",
    "evidence": [44]
  },
  "nda-5": {
    "label": "Entailment",
    "evidence": [55]
  },
  "nda-7": {
    "label": "Entailment",
    "evidence": [40]
  },
  "nda-8": {
    "label": "Entailment",
    "evidence": [71]
  },
  "nda-10": {
    "label": "Entailment",
    "evidence": [71]
  },
  "nda-11": {
    "label": "Entailment",
    "evidence": [53]
  },
  "nda-12": {
    "label": "Entailment",
    "evidence": [68]
  },
  "nda-13": {
    "label": "Entailment",
    "evidence": [68]
  },
  "nda-15": {
    "label": "Entailment",
    "evidence": [45]
  },
  "nda-16": {
    "label": "Entailment",
    "evidence": [60]
  },
  "nda-17": {
    "label": "Entailment",
    "evidence": [60]
  },
 

**How to read the two answers above.** Both models produce valid JSON with all 17 items — the strict
format was drilled in by the training data. The difference is in the *content*:

- **Labels:** the base model leans on one verdict ("Entailment" for almost everything — a classic
  base-model shortcut on legal text). The fine-tuned model distributes verdicts properly, including
  correct "NotMentioned" answers where the contract is silent.
- **Evidence:** the base model cites single, often-wrong clauses; the fine-tuned model cites precise
  multi-clause spans (e.g. `[84, 85, 86, 87, 88]` — the full paragraph that proves the point).

The scorer cell below turns this impression into numbers: **label accuracy** (right verdict per item)
and **evidence F1** (right clauses cited). On this run: base 0.565 / 0.063 → fine-tuned 0.600 / 0.098 —
better on both, and the workshop's full 123-contract pipeline shows how far the gap grows at scale
(64.5% → 87.4%).

> ⚠️ If a call ever times out ("invocation timed out…"), that contract made the model think past the
> endpoint's response window — re-run the cell or try `test_docs[1]`. Some long contracts trigger
> very long reasoning.

**How the scorer judges an answer:** for each of the 17 checklist items it checks two things —
did the model pick the **right verdict** (Entailment / Contradiction / NotMentioned), and did it cite the
**right evidence clauses** (overlap between predicted span numbers and the expert's spans, F1 = balance of
precision and recall). A model that says "Entailment" for everything gets some verdicts right but its
evidence F1 collapses — which is exactly how lazy answers get caught.

```mermaid
flowchart LR
    A["model's JSON answer<br/>for one contract"] --> B{"verdict matches<br/>expert?"}
    A --> C{"evidence spans<br/>overlap expert's?"}
    B --> D["label accuracy<br/>0 → 1 per item"]
    C --> E["evidence F1<br/>0 → 1 per item"]
    D & E --> F["average over N contracts<br/>per model, then compare"]
```

> 📊 **Small-sample honesty:** with N=5 contracts the numbers are noisy — treat the *direction*
> (fine-tuned ≥ base on both metrics) as the signal, and raise N for a stable estimate. The workshop's
> managed pipeline (123 contracts) measured 64.5% → 87.4%.

In [13]:
# ▶ RUN — score BOTH models on N contracts with the workshop's scorer logic (free, ~2–4 min)
import contractnli_scorer as S

N = 5          # contracts to score (raise to 10+ for a more stable estimate)

def gold_reference(d):
    return {k: {"label": v["choice"], "evidence": list(v["spans"])}
            for k, v in C.gold_for(d).items()}

def score_one(ep, ic, d, retries=3):
    b = dict(body, model_name=ic)
    for attempt in range(retries):
        try:
            r = smr.invoke_endpoint(EndpointName=ep, InferenceComponentName=ic,
                                    ContentType="application/json", Body=json.dumps(b))
            text = json.loads(r["Body"].read())["choices"][0]["message"]["content"]
            rec = {"model_response": text, "response": json.dumps(gold_reference(d))}
            out = {m["name"]: m["value"] for m in S.score_record(rec)["metrics_list"]}
            return out
        except Exception as e:
            print(f"  retry {attempt + 1} ({type(e).__name__})")
            time.sleep(20 * (attempt + 1))
    return {"label_correct": 0.0, "evidence_f1": 0.0, "json_valid": 0.0}

for which, ep, ic in [("BASE", BASE_EP, BASE_IC), ("SFT", EP_NAME, IC_NAME)]:
    rows = [score_one(ep, ic, d) for d in test_docs[:N]]
    lab = sum(r["label_correct"] for r in rows) / N
    f1 = sum(r["evidence_f1"] for r in rows) / N
    js = sum(r["json_valid"] for r in rows) / N
    print(f"{which:>4}: label accuracy {lab:.3f} | evidence F1 {f1:.3f} | valid JSON {js * 100:.0f}%")

print("\nReference (workshop, 123 contracts, managed pipelines): 64.5% -> 87.4% label accuracy")

BASE: label accuracy 0.565 | evidence F1 0.063 | valid JSON 100%
 SFT: label accuracy 0.600 | evidence F1 0.098 | valid JSON 100%

Reference (workshop, 123 contracts, managed pipelines): 64.5% -> 87.4% label accuracy


## Part 4 — Talk to it, production-style (endpoints already running)

Both endpoints from Part 3 are live. Now the moment every ML engineer waits for:
query the fine-tuned model like a real API, then **tear everything down** — an idle
endpoint is ~$1/hour, forever, until you delete it.

In [14]:
# ▶ RUN — ask your fine-tuned model about a REAL contract (pennies)
import re
from botocore.config import Config

C.ensure_dataset("./data")
doc, gold = test_docs[0], C.gold_for(test_docs[0])
smr = boto3.client("sagemaker-runtime", region_name=REGION,
                   config=Config(read_timeout=300, retries={"total_max_attempts": 3}))

body = {"model_name": IC_NAME,
        "messages": [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]}
                     for m in C.build_messages(doc, labels)],
        "max_tokens": 4000, "temperature": 0.0, "stop": ["<|im_end|>"], "stream": False}

resp = smr.invoke_endpoint(EndpointName=EP_NAME, InferenceComponentName=IC_NAME,
                           ContentType="application/json", Body=json.dumps(body))
text = json.loads(resp["Body"].read())["choices"][0]["message"]["content"]
print(text[:1500])

# quick validity check
clean = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
m = re.search(r"\{.*\}", clean, re.DOTALL)
print("\nvalid JSON verdict object:", bool(m))

<think>

</think>

{"nda-11": {"label": "NotMentioned", "evidence": []}, "nda-16": {"label": "Entailment", "evidence": [84, 85, 86, 87, 88]}, "nda-15": {"label": "Entailment", "evidence": [45, 46, 47]}, "nda-10": {"label": "NotMentioned", "evidence": []}, "nda-2": {"label": "NotMentioned", "evidence": []}, "nda-1": {"label": "NotMentioned", "evidence": []}, "nda-19": {"label": "Entailment", "evidence": [76]}, "nda-12": {"label": "NotMentioned", "evidence": []}, "nda-20": {"label": "Entailment", "evidence": [84, 85, 86, 87, 88]}, "nda-3": {"label": "NotMentioned", "evidence": []}, "nda-18": {"label": "NotMentioned", "evidence": []}, "nda-7": {"label": "Entailment", "evidence": [37, 40]}, "nda-17": {"label": "NotMentioned", "evidence": []}, "nda-8": {"label": "Entailment", "evidence": [71, 72]}, "nda-13": {"label": "Entailment", "evidence": [63, 66, 67]}, "nda-5": {"label": "Entailment", "evidence": [37, 40]}, "nda-4": {"label": "Entailment", "evidence": [44, 53]}}

valid JSON verdict ob

In [ ]:
# 🛑🛑 THE MOST IMPORTANT CELL FOR YOUR WALLET — ▶ RUN always, before you walk away
import time
time.sleep(45)   # let in-flight requests drain
# Order matters: delete children before parents (component → endpoint → config → model),
# like turning off the stove before leaving the kitchen. Billing stops the moment the
# ENDPOINT is gone — the rest is just tidiness.
for label, fn in [("fine-tuned IC", lambda: InferenceComponent.get(IC_NAME).delete()),
                  ("fine-tuned endpoint", lambda: Endpoint.get(EP_NAME).delete()),
                  ("fine-tuned config", lambda: EndpointConfig.get(CFG_NAME).delete()),
                  ("fine-tuned model", lambda: Model.get(MODEL_NAME).delete()),
                  ("base IC", lambda: InferenceComponent.get(BASE_IC).delete()),
                  ("base endpoint", lambda: Endpoint.get(BASE_EP).delete()),
                  ("base config", lambda: EndpointConfig.get(BASE_CFG).delete()),
                  ("base model", lambda: Model.get(BASE_MODEL_NAME).delete())]:
    try:
        fn(); print(f"deleted {label}")
    except Exception as e:
        print(f"skip {label}: {e}")
print("✅ TEARDOWN DONE — endpoint billing stopped")

## Part 5 — Teaching behavior: DPO (Direct Preference Optimization)

> 🎓 **The idea:** SFT teaches *facts and formats*; DPO teaches *taste*. The dataset is
> triplets — a `prompt`, a `chosen` response, and a `rejected` response. Training nudges the
> model toward chosen and away from rejected. No reward model, no RL loop — one elegant pass.
> When to use which: **SFT** for "answer in this JSON, cite this evidence"; **DPO** for
> tone, helpfulness, and how the model says "I don't know."

**The task:** make a tiny **Llama 3.2 1B Instruct** model sound human instead of
corporate-AI formal, using ~3,000 real human preference pairs.

In [16]:
# ▶ RUN — prepares preference data and registers it (~3 min, pennies)
import datasets
import pandas as pd
from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from sagemaker.train.dpo_trainer import DPOTrainer

DPO_MODEL = "meta-textgeneration-llama-3-2-1b-instruct"

stream = (load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train", streaming=True)
          .take(3000).shuffle(buffer_size=1000))
dset = datasets.Dataset.from_generator(lambda: stream, features=stream.features)
df = pd.DataFrame(dset)
train, val = train_test_split(df, test_size=0.2, random_state=42)
train, test = train_test_split(train, test_size=0.1, random_state=42)
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

def to_dpo(split, test=False):
    ds = Dataset.from_pandas(split, preserve_index=False)
    if test:
        return ds.map(lambda s: {"query": s["prompt"], "response": s["chosen"]},
                      remove_columns=list(ds.features))
    return ds.map(lambda s: {"prompt": s["prompt"], "chosen": s["chosen"], "rejected": s["rejected"]},
                  remove_columns=list(ds.features))

parts = {"train": to_dpo(train), "val": to_dpo(val), "test": to_dpo(test, test=True)}
dpo_paths = {}
for name, ds in parts.items():
    fname = f"humanlike_dpo_{name}.jsonl"
    p = pathlib.Path(f"./dpo_data/{name}"); p.mkdir(parents=True, exist_ok=True)
    ds.to_json(str(p / fname), orient="records")
    key = f"datasets/humanlike-dpo/{name}/{fname}"
    s3_client.upload_file(str(p / fname), BUCKET, key)
    dpo_paths[name] = f"s3://{BUCKET}/{key}"

dpo_train = DataSet.create(name="humanlike-dpo-train", source=dpo_paths["train"],
                           customization_technique=CustomizationTechnique.DPO,
                           wait=True, role=ROLE_ARN)
dpo_val = DataSet.create(name="humanlike-dpo-val", source=dpo_paths["val"],
                         customization_technique=CustomizationTechnique.DPO,
                         wait=True, role=ROLE_ARN)
DataSet.create(name="humanlike-dpo-test", source=dpo_paths["test"], wait=True, role=ROLE_ARN)
print("✅ DPO datasets registered")

                    INFO     TensorFlow version 2.19.1 available.                                     ]8;id=3235257;file:///opt/conda/lib/python3.12/site-packages/datasets/config.py\config.py]8;;\:]8;id=3235258;file:///opt/conda/lib/python3.12/site-packages/datasets/config.py#112\112]8;;\

README.md: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

train 2160 | val 600 | test 240


Map:   0%|          | 0/2160 [00:00<?, ? examples/s]

...

[10/05/26 17:17:48] INFO     Cannot simulate policies for                                  ]8;id=3235287;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3235288;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#429\429]8;;\
                             'arn:aws:iam::685057748560:role/service-role/AmazonSageMaker-                         
                             ExecutionRole-20261005T173732' (access denied); permission                            
                             verdict unknown.                                                                      

                    WARNING  Could not verify permissions for role                         ]8;id=3235293;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=3235294;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#658\658]8;;\
                             'arn:aws:iam::685057748560:role/service-role/AmazonSageMaker-                         
                             ExecutionRole-20261005T173732' (caller lacks                                          
                             iam:SimulatePrincipalPolicy). Proceeding with it. If the                              
                             operation later fails with an access-denied error, ensure the                         
                             role has the required permissions for 'training' (see                                 
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

Output()

Final Resource Status: Available

✅ DPO datasets registered


**How DPO learns from preferences:** each dataset row is a triplet. The model's probability of the
*chosen* answer is pushed up and the *rejected* answer pushed down — that's the entire algorithm. No reward
model, no RL loop, and still just LoRA adapters, so it's the same serverless one-call training.

```mermaid
flowchart LR
    P["💬 prompt<br/>'How do you make<br/>friends as an adult?'"] --> CH["✅ chosen<br/>warm, casual, asks<br/>a question back"]
    P --> RJ["❌ rejected<br/>'As an AI, I do not have<br/>personal experiences...'"]
    CH --> T["🎓 DPOTrainer<br/>P(chosen) ↑  P(rejected) ↓<br/>one serverless pass"]
    RJ --> T
```

In [ ]:
# 🛑 BEFORE YOU RUN — paid training (~$1–2, ~20 min). Only ONE serverless customization job
# can run at a time per region, so the SFT job must be finished (it is, if Part 2 completed).
# ▶ RUN — launches serverless DPO
candidate = f"{DPO_MODEL}-dpo"
MPG_DPO = candidate if len(candidate) <= 63 else candidate[:56] + "-dpo"

print("Step 1/2 📦 model package group — where the tuned weights will be registered...")
try:
    ModelPackageGroup.get(model_package_group_name=MPG_DPO)
except ClientError:
    ModelPackageGroup.create(model_package_group_name=MPG_DPO,
                             model_package_group_description="SageMaker serverless DPO")

print("Step 2/2 🚀 launching the serverless DPO job...")
# Same pattern as SFT: model id + dataset + LoRA. DPOTrainer instead of SFTTrainer — that's the diff.
dpo_trainer = DPOTrainer(
    model=DPO_MODEL,
    training_type=TrainingType.LORA,
    model_package_group=MPG_DPO,
    training_dataset=dpo_train,
    validation_dataset=dpo_val,
    s3_output_path=f"s3://{BUCKET}/{DPO_MODEL}-dpo",
    mlflow_experiment_name="humanlike-llama3-2-1b-dpo",
    base_job_name="dpo-llama32-1b",
    sagemaker_session=sess,
    accept_eula=True,
    role=ROLE_ARN,
)
dpo_job = dpo_trainer.train(wait=False)
DPO_JOB = dpo_job.training_job_name
print(f"LAUNCHED: {DPO_JOB}")

In [18]:
# ▶ RUN — watch the DPO job
while True:
    j = TrainingJob.get(training_job_name=DPO_JOB)
    print(f"{time.strftime('%H:%M:%S')}  {j.training_job_status} / {j.secondary_status}")
    if j.training_job_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(120)

17:17:53  InProgress / Starting
17:19:53  InProgress / Pending
17:21:53  InProgress / Downloading
17:23:53  InProgress / Downloading
17:25:53  InProgress / Training
17:27:53  InProgress / Training
17:29:53  InProgress / Training
17:31:53  InProgress / Training
17:33:53  InProgress / Training
17:35:53  InProgress / Training
17:37:53  InProgress / Training
17:39:53  InProgress / Training
17:41:54  InProgress / Uploading
17:43:54  Completed / Completed


### 5.1 — Measure it: the tone test

Same trick as Part 3, without any judge: deploy the DPO model and ask it something
casual. The base Llama 3.2 answers like a corporate assistant ("As an AI, I don't
have personal experiences…"); the DPO-tuned model should sound like a person — and
even ask you a question back. That difference is the entire point of preference
optimization, and you can *see* it without a single metric.

> 💡 **Want the base model on camera too?** The Llama weights are EULA-gated, so the
> easiest source of a base-model answer is the workshop's Lab 2 (it imports the base
> checkpoint for you). At scale, the LLM-as-judge path (custom rubrics like
> HumanLikeTone / ConversationalEngagement, judge model via Bedrock) is the managed
> option — see `scripts/06_evaluate_dpo.py`, which needs the evaluation quota from
> Part 0.

In [ ]:
# 🛑 BEFORE YOU RUN — third endpoint (~$1/hr; both previous endpoints were deleted in Part 4)
# ▶ RUN — deploy the DPO-tuned Llama 3.2 1B (~8–12 min)
from sagemaker.core.resources import ModelPackage

resp = sm.list_model_packages(ModelPackageGroupName=MPG_DPO, SortBy="CreationTime",
                              SortOrder="Descending", MaxResults=1)
dpo_pkg = ModelPackage.get(resp["ModelPackageSummaryList"][0]["ModelPackageArn"])
dpo_uri = sm_s3.s3_path_join(
    dpo_pkg.inference_specification.containers[0]
    .model_data_source.s3_data_source.s3_uri, "checkpoints", "hf_merged") + "/"

DPO_STEM = f"{DPO_MODEL[:38]}-dpo"
DPO_M_NAME, DPO_CFG, DPO_EP, DPO_IC = (f"{DPO_STEM}-{s}" for s in ["m", "cfg", "ep", "ic"])
dpo_env = {"SERVING_FAIL_FAST": "true", "OPTION_ASYNC_MODE": "true",
           "OPTION_ROLLING_BATCH": "disable", "OPTION_MAX_MODEL_LEN": "16384",
           "OPTION_TENSOR_PARALLEL_DEGREE": "max",
           "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
           "OPTION_TRUST_REMOTE_CODE": "true"}

print("Step 1/4 🧩 Model — packaging weights (S3) + serving container (DJL/vLLM)...")
Model.create(model_name=DPO_M_NAME,
             primary_container=ContainerDefinition(
                 image=IMAGE,
                 model_data_source=ModelDataSource(
                     s3_data_source=S3ModelDataSource(s3_uri=dpo_uri, s3_data_type="S3Prefix",
                                                      compression_type="None")),
                 environment=dpo_env),
             execution_role_arn=ROLE_ARN)
print(f"      ✅ model created")
print("Step 2/4 🖥  EndpointConfig — the hardware plan: ml.g5.xlarge × 1...")
EndpointConfig.create(endpoint_config_name=DPO_CFG, execution_role_arn=ROLE_ARN,
                      production_variants=[ProductionVariant(
                          variant_name="AllTraffic", instance_type="ml.g5.xlarge",
                          initial_instance_count=1,
                          model_data_download_timeout_in_seconds=700,
                          container_startup_health_check_timeout_in_seconds=700)])
print("      ✅ endpoint config created (billing starts at the next step)\n")
print("Step 3/4 ⚡ Endpoint — starting the GPU machine (~2 min)...")
Endpoint.create(endpoint_name=DPO_EP, endpoint_config_name=DPO_CFG)
Endpoint.get(DPO_EP).wait_for_status("InService")
print("      ✅ endpoint InService — the machine is up\n")
print("Step 4/4 📦 InferenceComponent — downloading ~8 GB of weights into the container")
print("      and loading them into vLLM — the long step, ~7–9 min. This is a good moment")
print("      to open the endpoint in the SageMaker console and watch it happen.")
InferenceComponent.create(
    inference_component_name=DPO_IC, endpoint_name=DPO_EP, variant_name="AllTraffic",
    specification=InferenceComponentSpecification(
        model_name=DPO_M_NAME,
        compute_resource_requirements=InferenceComponentComputeResourceRequirements(
            min_memory_required_in_mb=1024, number_of_accelerator_devices_required=1)),
    runtime_config=InferenceComponentRuntimeConfig(copy_count=1), region=REGION)
InferenceComponent.get(DPO_IC).wait_for_status("InService")
print("      ✅ inference component InService\n")
print(f"DPO model ready — sm:{DPO_EP}/{DPO_IC}@{REGION}")

In [20]:
# ▶ RUN — the tone test: one casual prompt, pennies, ~4 seconds
prompt = ("Hey, I just moved to a new city and don't really know anyone yet. "
          "How do you make friends as an adult?")
r = smr.invoke_endpoint(
    EndpointName=DPO_EP, InferenceComponentName=DPO_IC, ContentType="application/json",
    Body=json.dumps({"model_name": DPO_IC,
                     "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
                     "max_tokens": 400, "temperature": 0.7, "stream": False}))
print(json.loads(r["Body"].read())["choices"][0]["message"]["content"])

# Compare it against the BASE model's classic corporate answer:
print('\n--- base Llama 3.2 1B typically answers: ---')
print('"As an AI, I don\'t have personal experiences, but here are some '
      'general suggestions for making friends as an adult..."')
print("\nWarm, casual, asks a question back vs. robotic and corporate — that's DPO.")

 like, I've been there like, a million times! Moving to a new city can be so exciting, but it can also be super lonely. But don't worry, making friends is like, the best part of the whole experience, right? 😊 So, I've got a few tips that help me like, hit it off with people.

First of all, join like, a club or a group that's stuff you like, you know? Like, if you like hiking or trying new restaurants, join a hiking group or a foodie group. That way, you'll like, run into people who like the same stuff as you! 🏞️🍴

And, like, attend events and stuff like that. Concerts, festivals, that kind of thing. You never know who you'll meet there! 🎶🎉

Also, try like, to be open and like, friendly, you know? Don't be like, "Oh, I'm just going to this one party, and then I'm outta here." Like, actually go talk to people and like, start a conversation! 🤗

And, you know, don't be too hard on yourself if it doesn't like, fall right in at first. Making friends takes time, right? 😊 Like, it's just gonna

In [ ]:
# ▶ RUN — final teardown for this part (before you walk away)
time.sleep(45)   # let the last inference request drain before deleting
for label, fn in [("DPO IC", lambda: InferenceComponent.get(DPO_IC).delete()),
                  ("DPO endpoint", lambda: Endpoint.get(DPO_EP).delete()),
                  ("DPO config", lambda: EndpointConfig.get(DPO_CFG).delete()),
                  ("DPO model", lambda: Model.get(DPO_M_NAME).delete())]:
    try:
        fn(); print(f"deleted {label}")
    except Exception as e:
        print(f"skip {label}: {e}")
print("✅ DPO TEARDOWN DONE")

## 🎉 You did it — recap

| Part | What you learned | Real measured cost |
|---|---|---|
| 1–2 | SFT with LoRA, serverless, pay-per-token | ~$2, ~29–33 min |
| 3 | Evaluation without quota: side-by-side A/B with the workshop's own scorer | endpoint time only (~$1/hr, minutes) |
| 4 | vLLM deployment, OpenAI-schema inference, **teardown discipline** | ~$0.30 for the session |
| 5 | DPO + the tone test: preference optimization you can *see* | ~$1–2 train |

**Where next?** The fourth rung — **RLVR / RLAIF** (reinforcement learning with verifiable
rewards and AI feedback, including writing your *own* reward function) — is
[Lab 3 and Lab 4 of the AWS workshop](https://github.com/aws-samples/generative-ai-on-amazon-sagemaker/tree/main/workshops/serverless-model-customization-with-sagemaker-ai).
Run them the same way you ran this notebook.

**Cleanup checklist (back to ~$0):**
1. The teardown cells above deleted every endpoint ✔
2. Studio: stop the JupyterLab app (running apps bill hourly)
3. S3: `sagemaker-us-east-1-<account>` holds training outputs and datasets — empty it if you want literal zero
4. Optional: delete the Studio domain (Environment configuration → Domains)